## Foundry Agent with Code Interpreter Capability

![code_interpreter](./Images/code_interpreter.png)

#### Installing Required Libraries

In [1]:
%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity

Note: you may need to restart the kernel to use updated packages.


#### Setting up the Environment Variables

In [2]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, CodeInterpreterTool, CodeInterpreterToolAuto

load_dotenv()

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

print(f"Foundry Project Endpoint:", foundry_project_endpoint)
print(f"Model Deployment Name:", model_deployment_name)

Foundry Project Endpoint: https://skemuel-ai-901-proj-resource.services.ai.azure.com/api/projects/skemuel-ai-901-proj
Model Deployment Name: gpt-5.1


#### Setting up the Foundry Project Client

In [4]:
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

#### Creating the OpenAI Client

In [6]:
openai_client = project_client.get_openai_client()

#### CSV File Upload

In [7]:
# Upload the CSV file for the code interpreter to use
file = openai_client.files.create(purpose="assistants", file=open("./electronics_products.csv", "rb"))
print(f"File uploaded (id: {file.id})")

File uploaded (id: assistant-Nei1EUPP5ne2baFxAQ7v8x)


#### Creating Agent with the Code Interpreter Tool

In [8]:
agent_name = "code-interpreter-agent"

agent = project_client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=model_deployment_name,
        instructions="You are a helpful AI Assistant with code interpreter capabilities.",
        tools = [
            CodeInterpreterTool(
                container = CodeInterpreterToolAuto(
                    file_ids = [file.id]
                )
            )
        ]
    ),
)

# printing the agent id
print(f"Agent created (id: {agent.id}, name: {agent.name}, version: {agent.version})")

Agent created (id: code-interpreter-agent:1, name: code-interpreter-agent, version: 1)


#### Creating a Conversation Object for the Agent Chat System

In [9]:
# create a conversation to use with the agent
conversation = openai_client.conversations.create()
print(f"Created conversation with id: {conversation.id}")

Created conversation with id: conv_027002dc2224c8e3000a7jVe8n28mA2BQISOnONhgpIBLEPuWy


#### Calling our Agent

In [10]:
response = openai_client.responses.create(
    conversation = conversation.id,
    input = "Could you please create a column chart with products on the x-axis and their respective prices on the y-axis?",
    extra_body = {
        "agent": {
            "name": agent.name,
            "type": "agent_reference"
        }
    }
)

print(f"Response completed with id: {response.id}")

Response completed with id: resp_027002dc2224c8e3006adddcb331408190ae7abfa987904366


#### Extracting file information from response annotations

In [12]:
file_id = ""
filename = ""
container_id = ""

# Get the last message which should contain file citations
last_message = response.output[-1]  # ResponseOutputMessage
if last_message.type == "message":
        # Get the last content item (contains the file annotations)
        text_content = last_message.content[-1]  # ResponseOutputText
        if text_content.type == "output_text":
            # Get the last annotation (most recent file)
            if text_content.annotations:
                file_citation = text_content.annotations[-1]  # AnnotationContainerFileCitation
                if file_citation.type == "container_file_citation":
                    file_id = file_citation.file_id
                    filename = file_citation.filename
                    container_id = file_citation.container_id
                    print(f"Found generated file: {filename} (ID: {file_id})")

# Download the generated file if available
if file_id and filename:
        file_content = openai_client.containers.files.content.retrieve(file_id=file_id, container_id=container_id)
        with open(filename, "wb") as f:
            f.write(file_content.read())
            print(f"File {filename} downloaded successfully.")
        print(f"File ready for download: {filename}")
else:
        print("No file generated in response")

Found generated file: product_prices_column_chart.png (ID: cfile_6adddcb8a41c81908a22644db6c5104f)
File product_prices_column_chart.png downloaded successfully.
File ready for download: product_prices_column_chart.png
